## Routing

In [1]:
%pip install -U langchain-core langchain-ollama pydantic


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# Define the data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal[
        "python_docs",
        "js_docs",
        "golang_docs"
    ] = Field(
        description="Choose the most relevant datasource for the question."
    )

# Initialize local LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Configure structured output
structured_llm = llm.with_structured_output(
    RouteQuery,
    method="json_schema"
)

# Create prompt
system = """You are an expert at routing a user question
to the appropriate data source.

Based on the programming language mentioned,
route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "{question}")
])

# Create router
router = prompt | structured_llm

In [3]:
question = "How do I create a dictionary in Python?"

result = router.invoke({"question": question})

print(result)
print(result.datasource)

datasource='python_docs'
python_docs


In [4]:
from langchain_core.runnables import RunnableLambda

def choose_route(result):
    if "python_docs" in result.datasource.lower():
        return "chain for python_docs"

    elif "js_docs" in result.datasource.lower():
        return "chain for js_docs"

    else:
        return "golang_docs"

# Connect router with route selection
full_chain = router | RunnableLambda(choose_route)

In [5]:
question = "How do I create a dictionary in Python?"

response = full_chain.invoke({"question": question})

print(response)

chain for python_docs


## Semantic Routing

In [6]:
import numpy as np

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough
)

# Define two prompts
physics_template = """You are a very smart physics professor.
You explain physics concepts clearly and simply.
If you don't know an answer, admit it.

Here is a question:
{query}"""

math_template = """You are a very good mathematician.
You solve complex mathematical problems by breaking
them down into smaller, understandable steps.

Here is a question:
{query}"""

# Initialize the embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Generate embeddings for both prompts
prompt_templates = [physics_template, math_template]

prompt_embeddings = embeddings.embed_documents(
    prompt_templates
)

In [7]:
def prompt_router(input):

    # Embed the user's question
    query_embedding = embeddings.embed_query(
        input["query"]
    )

    # Compute cosine similarity
    query_vector = np.array(query_embedding)
    prompt_vectors = np.array(prompt_embeddings)

    similarity = (
        prompt_vectors @ query_vector
    ) / (
        np.linalg.norm(prompt_vectors, axis=1)
        * np.linalg.norm(query_vector)
    )

    # Find the most similar prompt
    most_similar = prompt_templates[
        np.argmax(similarity)
    ]

    print(
        "Using MATH"
        if most_similar == math_template
        else "Using PHYSICS"
    )

    return PromptTemplate.from_template(
        most_similar
    )

In [8]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

In [9]:
response = chain.invoke("What's a black hole?")
print(response)

Using PHYSICS
Black holes! One of the most fascinating and mysterious objects in the universe.

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. If you place a heavy object, like a bowling ball, on the trampoline, it will warp and curve, creating a dent. Now, imagine the bowling ball is so massive that it warps the entire trampoline, creating a massive gravitational field. That's roughly what happens with a black hole.

The point of no return, called the event horizon, marks the boundary of the black hole. Once you cross the event horizon, you're trapped by the black hole's gravity and can't escape. The gravity is so strong that it even warps the light around it, which is why we call it a "black" hole.

Now, here's the really cool (and mind-bendi

In [10]:
response = chain.invoke(
    "How do I solve a quadratic equation?"
)

print(response)

Using MATH
Solving quadratic equations is a fundamental concept in mathematics, and I'd be happy to break it down into smaller, manageable steps.

**Step 1: Understand the Quadratic Equation**

A quadratic equation is a polynomial equation of degree two, which means it has a squared variable (usually x). The general form of a quadratic equation is:

ax^2 + bx + c = 0

where a, b, and c are constants, and x is the variable.

**Step 2: Identify the Coefficients**

To solve the quadratic equation, we need to identify the coefficients a, b, and c. These coefficients are the numbers in front of the squared variable, the linear variable, and the constant term, respectively.

**Step 3: Choose a Method**

There are several methods to solve quadratic equations, including:

* Factoring (if possible)
* Completing the square
* Using the quadratic formula

For this example, let's use the quadratic formula, which is a widely used and effective method.

**Step 4: Apply the Quadratic Formula**

The qu

## Query Construction

In [12]:
import os
from getpass import getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = getpass("Enter LangSmith API key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-from-scratch"

In [20]:
%pip install -U youtube-transcript-api pytube


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
from langchain_community.document_loaders import YoutubeLoader

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4",
    add_video_info=False
).load()

print("Transcript:")
print(docs[0].page_content[:500])

print("\nMetadata:")
print(docs[0].metadata)

Transcript:
hi this is Lance from Lang chain I'm going to be talking about using Lang graph to build a diverse and sophisticated rag flows so just to set the stage the basic rag flow you can see here starts with a question retrieval of relevant documents from an index which are passed into the context window of an llm for generation of an answer grounded in the ret documents so that's kind of the basic outline and we can see it's like a very linear path um in practice though you often encounter a few differ

Metadata:
{'source': 'pbAd8O1Lvm4'}


In [18]:
import datetime
from typing import Optional
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos."""

    content_search: str = Field(
        description="Similarity search query applied to video transcripts."
    )

    title_search: str = Field(
        description="Short keyword search query applied to video titles."
    )

    min_view_count: Optional[int] = Field(
        default=None,
        description="Minimum view count, inclusive. Only if specified."
    )

    max_view_count: Optional[int] = Field(
        default=None,
        description="Maximum view count, exclusive. Only if specified."
    )

    earliest_publish_date: Optional[datetime.date] = Field(
        default=None,
        description="Earliest publication date, inclusive. Only if specified."
    )

    latest_publish_date: Optional[datetime.date] = Field(
        default=None,
        description="Latest publication date, exclusive. Only if specified."
    )

    min_length_sec: Optional[int] = Field(
        default=None,
        description="Minimum video duration in seconds."
    )

    max_length_sec: Optional[int] = Field(
        default=None,
        description="Maximum video duration in seconds."
    )

In [32]:
from langchain_core.prompts import ChatPromptTemplate

system = """
You convert natural language questions into structured
search queries for tutorial videos.

Extract search terms separately from metadata filters.

Rules:
- content_search: Only the topic being searched.
- title_search: Empty string unless the user specifically
  asks for a keyword in the title.
- For "published in YYYY":
  earliest_publish_date = YYYY-01-01
  latest_publish_date = (YYYY+1)-01-01
- Unspecified filters must be null.
- Never copy the entire question into search fields.

Example 1:
Question: Find Python tutorials published in 2022.
Output:
content_search = "Python tutorials"
title_search = ""
earliest_publish_date = "2022-01-01"
latest_publish_date = "2023-01-01"

Example 2:
Question: Find videos with RAG in the title.
Output:
content_search = "RAG"
title_search = "RAG"
earliest_publish_date = null
latest_publish_date = null

Example 3:
Question: Find LangChain videos shorter than 5 minutes.
Output:
content_search = "LangChain"
title_search = ""
max_length_sec = 300

Return all required fields according to the schema.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "{question}")
])

query_analyzer = prompt | structured_llm

In [33]:
query = query_analyzer.invoke({
    "question": "rvideos on chat langchain published in 2023"
})

print(query.model_dump())

{'content_search': 'videos on chat langchain', 'title_search': '', 'min_view_count': None, 'max_view_count': None, 'earliest_publish_date': datetime.date(2023, 1, 1), 'latest_publish_date': datetime.date(2024, 1, 1), 'min_length_sec': None, 'max_length_sec': None}


In [34]:
for key, value in query.model_dump().items():
    print(f"{key}: {value}")

content_search: videos on chat langchain
title_search: 
min_view_count: None
max_view_count: None
earliest_publish_date: 2023-01-01
latest_publish_date: 2024-01-01
min_length_sec: None
max_length_sec: None
